# NB13 — Background swap, expert dermoscopic structures, diagnostic coverage
**Settings:** Accelerator **GPU T4 x2** (one GPU is used) · Internet **On** (ISIC 2018 Task 2 ground truth is downloaded
from the public ISIC archive). **Inputs:** as NB11. **Run:** about 1.5 h.

**Pre-specified primary outcomes** (fixed before the run; everything else is secondary):
* **(A)** BG-Gap = AUC(Mixed-Same) - AUC(Mixed-Rand) (Xiao et al., ICLR 2021), lower = decisions follow the lesion, not the
  background; seed ensembles (paired bootstrap over lesion groups) and across runs (seven runs of PPN, Correia, Ours);
* **(B)** within-lesion excess of the hot region on expert-annotated dermoscopic structures: share of the on-lesion hot region
  that lies on an expert structure minus the share of the lesion those structures cover (0 = chance), validation and test
  images only (higher = attention inside the lesion goes to diagnostic structures, not merely to the lesion);
* **(C)** diagnostic coverage of the benign prototypes: mean over the seven benign diagnoses of the best one-vs-rest AUC
  of any benign prototype within benign test images (higher = the benign prototypes represent more diagnoses).

Comparisons: Ours against PPN-std, PPN, PPN-FA, IAIA-BL and the Correia formulation.

In [ ]:
%%writefile artproto.py
"""
artproto.py -- shared library for the paper
"Artifact-aware prototype learning for skin-lesion classification".

Everything (data caching, lesion/artifact masks, models, losses, training,
push, evaluation, counterfactuals) lives here so that every notebook uses
exactly the same code.
"""
import os, re, json, glob, math, time, random, copy, traceback
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

IMG = 256          # cached resolution (square, as in the original project)
OUT = 224          # network input resolution
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# bit layout of the uint8 mask array
BIT_L, BIT_DC, BIT_TS, BIT_SP, BIT_INK = 1, 2, 4, 8, 16
BIT_ART = BIT_DC | BIT_TS | BIT_SP | BIT_INK
ART_BITS = {"DC": BIT_DC, "TS": BIT_TS, "SP": BIT_SP, "INK": BIT_INK}
ART_LONG = {"DC": "dark corner / vignette", "TS": "thin dark structures (hair, ruler ticks)",
            "SP": "specular highlights / gel bubbles", "INK": "ink markings / coloured patches"}
DETECTOR_VERSION = 2
# image-level presence thresholds (fraction of image area)
ART_MIN_FRAC = {"DC": 0.01, "TS": 0.003, "SP": 0.0005, "INK": 0.001}


# ----------------------------------------------------------------------------- utils
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def walk_files(root="/kaggle/input"):
    """Recursive file list. followlinks=True is REQUIRED: recent Kaggle images mount
    datasets as symlinks under /kaggle/input, which os.walk skips by default."""
    out, seen = [], set()
    for r, ds, fs in os.walk(root, followlinks=True):
        real = os.path.realpath(r)
        if real in seen:                      # guard against symlink cycles
            ds[:] = []; continue
        seen.add(real)
        for f in fs:
            out.append(os.path.join(r, f).replace(os.sep, '/'))
    return out


def load_rgb_resized(path, size=IMG):
    from PIL import Image
    im = Image.open(path)
    try:
        im.draft("RGB", (size * 2, size * 2))      # fast JPEG decoding at reduced scale
    except Exception:
        pass
    a = np.asarray(im.convert("RGB"))
    return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)


def load_mask_resized(path, size=IMG):
    from PIL import Image
    m = np.asarray(Image.open(path).convert("L"))
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
    return (m >= 128).astype(np.uint8)


def phash64(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA)
    d = cv2.dct(g)[:8, :8].flatten()
    return d > np.median(d[1:])


def thumbs(img):
    """z-normalised 32x32 grey thumbnails of the full image and of the central 50% crop
    (used to VERIFY pHash duplicate candidates; the centre crop ignores vignettes)."""
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    out = []
    for a in (g, g[64:192, 64:192]):
        t = cv2.resize(a, (32, 32), interpolation=cv2.INTER_AREA).flatten()
        t = (t - t.mean()) / (t.std() + 1e-6) / np.sqrt(t.size)
        out.append(t.astype(np.float16))
    return out[0], out[1]


def hamming_pairs(bitsA, bitsB=None, thr=6, device="cuda", chunk=2048):
    """Return (i, j, ham) with ham <= thr. If bitsB is None: pairs inside A with i<j."""
    dev = device if torch.cuda.is_available() else "cpu"
    dt = torch.float16 if dev != "cpu" else torch.float32
    A = torch.from_numpy(bitsA.astype(np.float32) * 2 - 1).to(dev, dt)
    B = A if bitsB is None else torch.from_numpy(bitsB.astype(np.float32) * 2 - 1).to(dev, dt)
    out = []
    for s in range(0, len(A), chunk):
        ham = (64 - (A[s:s + chunk] @ B.T).float()) / 2
        ii, jj = torch.nonzero(ham <= thr + 1e-3, as_tuple=True)
        h = ham[ii, jj]
        ii = ii + s
        if bitsB is None:
            keep = ii < jj
            ii, jj, h = ii[keep], jj[keep], h[keep]
        out.append(torch.stack([ii.float(), jj.float(), h], 1).cpu())
    o = torch.cat(out).numpy() if out else np.zeros((0, 3))
    return o[:, 0].astype(int), o[:, 1].astype(int), o[:, 2].astype(int)


class UnionFind:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)


# ----------------------------------------------------------------------------- masks
def postprocess_lesion(prob, thr=0.5, min_frac=0.005):
    m = (prob >= thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.zeros_like(m), False
    j = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    m = (lab == j).astype(np.uint8)
    inv = 1 - m                                         # fill holes
    n2, lab2, _, _ = cv2.connectedComponentsWithStats(inv, 4)
    border = np.unique(np.concatenate([lab2[0], lab2[-1], lab2[:, 0], lab2[:, -1]]))
    holes = ~np.isin(lab2, border) & (inv > 0)
    m[holes] = 1
    return m, bool(m.mean() >= min_frac)


def otsu_lesion(img):
    """Classical fallback lesion segmentation (only used if no GT masks are available)."""
    b = cv2.GaussianBlur(img[..., 2], (11, 11), 0)
    _, m = cv2.threshold(b, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return postprocess_lesion(m.astype(np.float32))


def detect_artifacts(img, L):
    """Rule-based artifact masks on a 256x256 RGB uint8 image.
    L: uint8 lesion mask. Artifact pixels are never inside the (dilated) lesion.
    Returns uint8 bitfield (DC|TS|SP|INK)."""
    H, W = L.shape
    area = H * W
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    Hh, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    Ld = cv2.dilate(L.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))) > 0

    # (1) dark corners / vignette: dark regions connected to the image border
    dark = cv2.morphologyEx((V < 50).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(dark, 8)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    # must be a large blob (area) and not a thin line touching the border (hair): mean thickness >= 8 px
    keep = [b for b in border if b != 0 and st[b, cv2.CC_STAT_AREA] >= 0.002 * area and
            st[b, cv2.CC_STAT_AREA] / max(np.hypot(st[b, cv2.CC_STAT_WIDTH], st[b, cv2.CC_STAT_HEIGHT]), 1) >= 8]
    dc = np.isin(lab, keep) & ~Ld

    # (2) thin dark structures (hair, ruler ticks/lines): black-hat + elongation filter
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    bh = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, k)
    ts = (bh >= 18).astype(np.uint8)
    ts[cv2.dilate(dc.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0] = 0
    ts[:4, :] = 0; ts[-4:, :] = 0; ts[:, :4] = 0; ts[:, -4:] = 0     # v2: no frame-edge strips
    n, lab, st, _ = cv2.connectedComponentsWithStats(ts, 8)
    a = st[:, cv2.CC_STAT_AREA].astype(np.float64)
    diag = np.hypot(st[:, cv2.CC_STAT_WIDTH], st[:, cv2.CC_STAT_HEIGHT])
    good = (a >= 15) & (diag >= 15) & (a / np.maximum(diag, 1) <= 6.0)
    good[0] = False
    ts = good[lab] & ~Ld

    # (3) specular highlights / gel bubbles: small bright unsaturated blobs that are much brighter
    #     than their surround (v2: white top-hat keeps only bright peaks smaller than 11 px -> large
    #     whitish skin areas are rejected; bubble rims and glints are kept)
    tophat = cv2.morphologyEx(V, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
    sp = ((V >= 210) & (S <= 50) & (tophat >= 25)).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(sp, 8)
    a = st[:, cv2.CC_STAT_AREA]
    good = (a >= 3) & (a <= 0.01 * area)
    good[0] = False
    sp = (cv2.dilate(good[lab].astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & ~Ld

    # (4) ink markings / coloured patches: strongly saturated blue-cyan-violet regions outside the lesion
    #     (v2: hue 90-150 also covers blue/cyan stickers; S>=90 rejects bluish-tinted skin; pink skin
    #      (H>150) excluded; a 12-px band next to the vignette is excluded (chromatic fringe))
    ink = cv2.morphologyEx(((Hh >= 90) & (Hh <= 150) & (S >= 90) & (V >= 40) & (V <= 235)).astype(np.uint8),
                           cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    good = st[:, cv2.CC_STAT_AREA] >= 40
    good[0] = False
    ink = good[lab] & ~Ld & ~(cv2.dilate(dc.astype(np.uint8), np.ones((25, 25), np.uint8)) > 0)

    return (dc * BIT_DC | ts * BIT_TS | sp * BIT_SP | ink * BIT_INK).astype(np.uint8)


def artifact_flags(mask_bits):
    """image-level presence flags from a (256,256) bitfield."""
    return {k: float(((mask_bits & b) > 0).mean()) >= ART_MIN_FRAC[k] for k, b in ART_BITS.items()}


# ----------------------------------------------------------------------------- GPU augmentation
def gpu_augment(img_u8, msk_u8, mode, out=OUT, gen=None):
    """img_u8 (B,256,256,3) uint8, msk_u8 (B,256,256) uint8 (on device).
    mode: 'eval' | 'std' | 'robust'. Image and masks receive the SAME geometric transform.
    Returns normalized image (B,3,out,out), lesion mask, artifact mask (B,1,out,out) in [0,1]."""
    dev = img_u8.device
    B = img_u8.shape[0]
    x = img_u8.permute(0, 3, 1, 2).float() / 255.0
    m = msk_u8.long()
    mL = ((m & BIT_L) > 0).float()[:, None]
    mA = ((m & BIT_ART) > 0).float()[:, None]
    R = lambda *s: torch.rand(*s, device=dev, generator=gen)
    theta = torch.zeros(B, 2, 3, device=dev)
    if mode == "eval":
        theta[:, 0, 0] = 1; theta[:, 1, 1] = 1
    elif mode == "std":       # Resize(256) + RandomCrop(224) + HFlip
        s = out / IMG
        fx = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = s * fx; theta[:, 1, 1] = s
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - s); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - s)
    elif mode == "robust":    # RandomResizedCrop(scale .4-1, ratio 3/4-4/3) + H/V flips (+ colour jitter)
        area = 0.4 + 0.6 * R(B)
        logr = math.log(3 / 4) + (math.log(4 / 3) - math.log(3 / 4)) * R(B)
        r = torch.exp(logr)
        sx = torch.sqrt(area * r).clamp(max=1.0); sy = torch.sqrt(area / r).clamp(max=1.0)
        fx = torch.where(R(B) < 0.5, -1.0, 1.0); fy = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = sx * fx; theta[:, 1, 1] = sy * fy
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - sx); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - sy)
    else:
        raise ValueError(mode)
    grid = F.affine_grid(theta, (B, 3, out, out), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    mm = F.grid_sample(torch.cat([mL, mA], 1), grid, mode="bilinear", padding_mode="border", align_corners=False)
    mL, mA = mm[:, :1], mm[:, 1:]
    if mode == "robust":      # ColorJitter(brightness=.2, contrast=.2, saturation=.2)
        b = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        x = (x * b).clamp(0, 1)
        c = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).mean((1, 2)).view(B, 1, 1, 1)
        x = ((x - g) * c + g).clamp(0, 1)
        s = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        gy = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2])[:, None]
        x = (gy + (x - gy) * s).clamp(0, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std, mL, mA


# ----------------------------------------------------------------------------- models
def densenet_features():
    from torchvision.models import densenet121, DenseNet121_Weights
    return densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1).features


class DenseCNN(nn.Module):
    """Black-box reference: DenseNet-121 + GAP + linear."""
    kind = "cnn"
    def __init__(self, n_classes=2, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.head = nn.Linear(1024, n_classes)
    def forward(self, x):
        f = F.relu(self.features(x))
        return {"logits": self.head(f.mean((2, 3))), "fmap": f}


class ProtoPNet(nn.Module):
    """ProtoPNet (Chen et al., NeurIPS 2019) with a DenseNet-121 backbone."""
    kind = "proto"
    def __init__(self, n_classes=2, ppc=10, D=128, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.add_on = nn.Sequential(nn.Conv2d(1024, D, 1), nn.ReLU(), nn.Conv2d(D, D, 1), nn.Sigmoid())
        P = n_classes * ppc
        self.prototypes = nn.Parameter(torch.rand(P, D, 1, 1))
        self.register_buffer("proto_class", torch.arange(P) // ppc)
        ident = F.one_hot(torch.arange(P) // ppc, n_classes).float().T          # (C,P)
        self.register_buffer("class_identity", ident)
        self.last = nn.Linear(P, n_classes, bias=False)
        with torch.no_grad():
            self.last.weight.copy_(1.0 * ident - 0.5 * (1 - ident))
        self.n_classes, self.P, self.D = n_classes, P, D

    def latent(self, x):
        f = F.relu(self.features(x))
        with torch.autocast("cuda", enabled=False):
            return self.add_on(f.float())

    def distances(self, z):
        x2 = (z ** 2).sum(1, keepdim=True)
        p2 = (self.prototypes ** 2).sum((1, 2, 3)).view(1, -1, 1, 1)
        xp = F.conv2d(z, self.prototypes)
        return F.relu(x2 + p2 - 2 * xp)

    def forward(self, x):
        z = self.latent(x)
        with torch.autocast("cuda", enabled=False):
            d = self.distances(z)
            sim = torch.log((d + 1) / (d + 1e-4))
            act = sim.amax((2, 3))
            logits = self.last(act)
        return {"logits": logits, "sim": sim, "dist": d, "min_d": d.amin((2, 3)), "act": act, "z": z}


def _tiny_features():  # CPU unit tests only
    return nn.Sequential(nn.Conv2d(3, 1024, 32, stride=32))


def build_model(cfg, pretrained=True):
    if cfg["model"] == "cnn":
        return DenseCNN(pretrained=pretrained)
    return ProtoPNet(ppc=cfg.get("ppc", 10), D=cfg.get("D", 128), pretrained=pretrained)


# ----------------------------------------------------------------------------- losses
def proto_base_losses(out, y, model, class_w):
    ce = F.cross_entropy(out["logits"], y, weight=class_w)
    md = out["min_d"]
    own = model.proto_class[None, :] == y[:, None]
    big = torch.finfo(md.dtype).max / 4
    clst = md.masked_fill(~own, big).amin(1).mean()
    sep = md.masked_fill(own, big).amin(1).mean()
    return ce, clst, sep


def artifact_lesion_losses(out, y, model, cL, cA, valid, tau=0.5):
    """Proposed losses.  sim: (B,P,h,w); cL/cA: lesion/artifact fraction per latent cell (B,1,h,w).
    L_art : activation-weighted spatial-attention mass that every prototype puts on artifact cells.
    L_les : -log attention mass that own-class prototypes put on lesion cells."""
    sim = out["sim"]
    B, P, h, w = sim.shape
    att = F.softmax(sim.flatten(2) / tau, dim=2)                         # (B,P,hw)
    wt = sim.flatten(2).amax(2).detach()
    wt = wt / wt.sum(1, keepdim=True).clamp_min(1e-8)
    a_art = (att * cA.flatten(2)).sum(2)                                  # (B,P)
    l_art_i = (wt * a_art).sum(1)
    own = (model.proto_class[None, :] == y[:, None]).float()
    w_own = wt * own
    w_own = w_own / w_own.sum(1, keepdim=True).clamp_min(1e-8)
    a_les = (att * cL.flatten(2)).sum(2)
    l_les_i = (w_own * -torch.log(a_les + 1e-4)).sum(1)
    has_les = valid & (cL.flatten(1).sum(1) >= 0.5)
    zero = sim.sum() * 0
    l_art = l_art_i[valid].mean() if valid.any() else zero
    l_les = l_les_i[has_les].mean() if has_les.any() else zero
    return l_art, l_les, a_art.detach(), a_les.detach()


def fine_annotation_loss(out, y, model, cL, valid):
    """IAIA-BL style fine-annotation loss (Barnett et al., 2021) adapted to lesion masks:
    own-class prototypes are penalised for activating outside the lesion, other-class
    prototypes for activating anywhere."""
    sim = out["sim"]
    own = model.proto_class[None, :] == y[:, None]
    t_own = (sim * (1 - cL)).flatten(2).norm(dim=2)
    t_oth = sim.flatten(2).norm(dim=2)
    per = (t_own * own).sum(1) + (t_oth * ~own).sum(1)
    return per[valid].mean() if valid.any() else sim.sum() * 0


def cell_fractions(mL, mA, hw):
    return F.adaptive_avg_pool2d(mL, hw), F.adaptive_avg_pool2d(mA, hw)


# ----------------------------------------------------------------------------- data
class ArrDS(torch.utils.data.Dataset):
    def __init__(self, img_path, msk_path, labels, valid, idx):
        self.img_path, self.msk_path = img_path, msk_path
        self.labels = np.asarray(labels).astype(np.int64)
        self.valid = np.asarray(valid).astype(bool)
        self.idx = np.asarray(idx)
        self.imgs = self.msks = None
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        if self.imgs is None:
            self.imgs = np.load(self.img_path, mmap_mode="r")
            self.msks = np.load(self.msk_path, mmap_mode="r")
        i = int(self.idx[k])
        return (torch.from_numpy(np.ascontiguousarray(self.imgs[i])),
                torch.from_numpy(np.ascontiguousarray(self.msks[i])),
                int(self.labels[i]), bool(self.valid[i]), i)


def make_loader(ds, bs, shuffle, seed=0, workers=4, drop_last=False):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=workers,
                                       pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                                       persistent_workers=workers > 0)


# ----------------------------------------------------------------------------- inference
@torch.no_grad()
def predict(model, loader, device, keep_maps=False):
    model.eval()
    probs, ys, ids, maps, acts = [], [], [], [], []
    for img, msk, y, v, i in loader:
        x, _, _ = gpu_augment(img.to(device, non_blocking=True), msk.to(device, non_blocking=True), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model(x.contiguous(memory_format=torch.channels_last))
        probs.append(F.softmax(out["logits"].float(), 1)[:, 1].cpu()); ys.append(y); ids.append(i)
        if keep_maps and "sim" in out:
            maps.append(out["sim"].half().cpu()); acts.append(out["act"].cpu())
    r = {"prob": torch.cat(probs).numpy(), "y": torch.cat(ys).numpy(), "idx": torch.cat(ids).numpy()}
    if keep_maps and maps:
        r["sim"] = torch.cat(maps).numpy(); r["act"] = torch.cat(acts).numpy()
    return r


def auc_safe(y, p):
    from sklearn.metrics import roc_auc_score
    try:
        return float(roc_auc_score(y, p))
    except Exception:
        return float("nan")


# ----------------------------------------------------------------------------- push
@torch.no_grad()
def push_prototypes(model, loader, device, constrained, les_min=0.5, art_max=0.05):
    """Project every prototype onto its nearest latent patch of a training image of its class.
    constrained=True: candidate patches must lie on the lesion (lesion fraction >= les_min)
    and be artifact-free (artifact fraction <= art_max)."""
    model.eval()
    P, D = model.P, model.D
    best_d = torch.full((P,), float("inf"), device=device)
    best_z = model.prototypes.data.view(P, D).clone()
    info = [dict(proto=j, cls=int(model.proto_class[j]), img=-1, h=-1, w=-1, dist=float("inf"),
                 les_frac=float("nan"), art_frac=float("nan")) for j in range(P)]
    for img, msk, y, v, ii in loader:
        img, msk, y, v = img.to(device), msk.to(device), y.to(device), v.to(device)
        x, mL, mA = gpu_augment(img, msk, "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z = model.latent(x.contiguous(memory_format=torch.channels_last))
        d = model.distances(z.float())                                    # (B,P,h,w)
        B, _, h, w = d.shape
        cL, cA = cell_fractions(mL, mA, (h, w))
        ok = torch.ones(B, 1, h, w, dtype=torch.bool, device=device)
        if constrained:
            ok = (cL >= les_min) & (cA <= art_max) & v.view(B, 1, 1, 1)
        own = model.proto_class.view(1, P, 1, 1) == y.view(B, 1, 1, 1)
        dd = d.masked_fill(~(own & ok), float("inf")).permute(1, 0, 2, 3).reshape(P, -1)
        val, arg = dd.min(1)
        upd = val < best_d
        for j in torch.nonzero(upd).flatten().tolist():
            a = int(arg[j]); b, rem = divmod(a, h * w); hh, ww = divmod(rem, w)
            best_d[j] = val[j]; best_z[j] = z[b, :, hh, ww].float()
            info[j].update(img=int(ii[b]), h=hh, w=ww, dist=float(val[j]),
                           les_frac=float(cL[b, 0, hh, ww]), art_frac=float(cA[b, 0, hh, ww]))
    model.prototypes.data.copy_(best_z.view(P, D, 1, 1))
    return pd.DataFrame(info)


# ----------------------------------------------------------------------------- training
DEFAULT = dict(model="proto", aug="robust", lam_clst=0.8, lam_sep=0.08, lam_l1=1e-4,
               lam_art=0.0, lam_les=0.0, lam_fa=0.0, tau=0.5, push="standard",
               ep_warm=5, ep_joint=15, ep_ll=5, bs=64, lr_feat=1e-4, lr_addon=3e-3, lr_proto=3e-3,
               lr_head=1e-3, lr_ll=1e-3, wd=1e-3, seed=0)

CONFIGS = {
    "CNN":         dict(model="cnn"),
    "PPN-std":     dict(aug="std"),
    "PPN":         dict(),
    "PPN-FA":      dict(lam_fa=1e-3),
    "Ours":        dict(lam_art=1.0, lam_les=1.0, push="constrained"),
    "Ours-noLes":  dict(lam_art=1.0, lam_les=0.0, push="constrained"),
    "Ours-noArt":  dict(lam_art=0.0, lam_les=1.0, push="constrained"),
    "Ours-stdPush": dict(lam_art=1.0, lam_les=1.0, push="standard"),
}


def full_cfg(name, seed):
    c = dict(DEFAULT); c.update(CONFIGS[name]); c["name"] = name; c["seed"] = seed
    return c


def _set_trainable(model, stage):
    for p in model.parameters(): p.requires_grad = False
    if model.kind == "cnn":
        for p in model.head.parameters(): p.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
        return
    if stage in ("warm", "joint"):
        for p in model.add_on.parameters(): p.requires_grad = True
        model.prototypes.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
    elif stage == "last":
        for p in model.last.parameters(): p.requires_grad = True


def train_run(cfg, prep, outdir, device, smoke=False, workers=4, log=print):
    """prep: dict(img_path, msk_path, meta DataFrame with split,label,lesion_valid)."""
    os.makedirs(outdir, exist_ok=True)
    set_seed(cfg["seed"])
    torch.backends.cudnn.benchmark = True
    meta = prep["meta"]
    tr = np.where(meta.split.values == "train")[0]
    va = np.where(meta.split.values == "val")[0]
    if smoke:
        rs = np.random.RandomState(0)
        tr, va = rs.choice(tr, min(512, len(tr)), replace=False), rs.choice(va, min(256, len(va)), replace=False)
        cfg = dict(cfg, ep_warm=1, ep_joint=1, ep_ll=1)
    labels, valid = meta.label.values, meta.lesion_valid.values
    mk = lambda idx: ArrDS(prep["img_path"], prep["msk_path"], labels, valid, idx)
    tr_loader = make_loader(mk(tr), cfg["bs"], True, cfg["seed"], workers, drop_last=True)
    tr_eval_loader = make_loader(mk(tr), 128, False, 0, workers)
    va_loader = make_loader(mk(va), 128, False, 0, workers)
    n1 = labels[tr].sum(); n0 = len(tr) - n1
    class_w = torch.tensor([len(tr) / (2 * n0), len(tr) / (2 * n1)], dtype=torch.float32, device=device)

    model = build_model(cfg, pretrained=prep.get("pretrained", True)).to(device).to(memory_format=torch.channels_last)
    is_proto = model.kind == "proto"
    if is_proto:
        opt_warm = torch.optim.Adam([{"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                     {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                      {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
    else:
        opt_warm = torch.optim.Adam(model.head.parameters(), lr=cfg["lr_head"])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.head.parameters(), "lr": cfg["lr_head"]}])
    sched = torch.optim.lr_scheduler.StepLR(opt_joint, step_size=5, gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    hist, best = [], dict(auc=-1, state=None, epoch=-1)
    E = cfg["ep_warm"] + cfg["ep_joint"]
    t_start = time.time()

    def one_epoch(opt, stage):
        # last-layer stage: everything below the last layer is frozen INCLUDING BatchNorm statistics,
        # so the latent space (and hence the pushed prototypes = real training patches) stays exact.
        model.train(stage != "last")
        agg = {}
        for img, msk, y, v, ii in tr_loader:
            img, msk = img.to(device, non_blocking=True), msk.to(device, non_blocking=True)
            y, v = y.to(device), v.to(device)
            x, mL, mA = gpu_augment(img, msk, cfg["aug"])
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                out = model(x.contiguous(memory_format=torch.channels_last))
            parts = {}
            if is_proto:
                ce, clst, sep = proto_base_losses(out, y, model, class_w)
                parts.update(ce=ce, clst=clst, sep=sep)
                if stage == "last":
                    l1 = (model.last.weight * (1 - model.class_identity)).norm(p=1)
                    loss = ce + cfg["lam_l1"] * l1
                else:
                    loss = ce + cfg["lam_clst"] * clst - cfg["lam_sep"] * sep
                    if cfg["lam_art"] > 0 or cfg["lam_les"] > 0:
                        cL, cA = cell_fractions(mL, mA, out["sim"].shape[2:])
                        la, ll, _, _ = artifact_lesion_losses(out, y, model, cL, cA, v, cfg["tau"])
                        loss = loss + cfg["lam_art"] * la + cfg["lam_les"] * ll
                        parts.update(art=la, les=ll)
                    if cfg["lam_fa"] > 0:
                        cL, _ = cell_fractions(mL, mA, out["sim"].shape[2:])
                        fa = fine_annotation_loss(out, y, model, cL, v)
                        loss = loss + cfg["lam_fa"] * fa
                        parts.update(fa=fa)
            else:
                loss = F.cross_entropy(out["logits"].float(), y, weight=class_w)
                parts.update(ce=loss)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            for k, t in parts.items():
                agg[k] = agg.get(k, 0.0) + float(t.detach())
            agg["n"] = agg.get("n", 0) + 1
        n = agg.pop("n", 1)
        return {k: round(t / n, 4) for k, t in agg.items()}

    for ep in range(1, E + 1):
        stage = "warm" if ep <= cfg["ep_warm"] else "joint"
        _set_trainable(model, stage)
        t0 = time.time()
        tl = one_epoch(opt_warm if stage == "warm" else opt_joint, stage)
        if stage == "joint": sched.step()
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        rec = dict(epoch=ep, stage=stage, val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
        hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
        if not is_proto and auc > best["auc"]:
            best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=ep)

    push_info = None
    if is_proto:
        push_info = push_prototypes(model, tr_eval_loader, device, constrained=cfg["push"] == "constrained")
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        hist.append(dict(epoch=E, stage="after_push", val_auc=round(auc, 4)))
        log(f"[{cfg['name']} s{cfg['seed']}] after push val_auc={auc:.4f} | protos on artifact cells(>0.5): "
            f"{int((push_info.art_frac > 0.5).sum())}/{len(push_info)} | on lesion cells(>=0.5): "
            f"{int((push_info.les_frac >= 0.5).sum())}/{len(push_info)}")
        best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=E)
        _set_trainable(model, "last")
        opt_ll = torch.optim.Adam(model.last.parameters(), lr=cfg["lr_ll"])
        for k in range(1, cfg["ep_ll"] + 1):
            t0 = time.time()
            tl = one_epoch(opt_ll, "last")
            pv = predict(model, va_loader, device)
            auc = auc_safe(pv["y"], pv["prob"])
            rec = dict(epoch=E + k, stage="last", val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
            hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
            if auc > best["auc"]:
                best = dict(auc=auc, state=copy.deepcopy({k2: t.cpu() for k2, t in model.state_dict().items()}), epoch=E + k)
        push_info.to_csv(os.path.join(outdir, "push_info.csv"), index=False)

    if best["state"] is None:   # e.g. undefined val AUC: never save an empty checkpoint
        best = dict(auc=float("nan"), state={k: t.cpu() for k, t in model.state_dict().items()}, epoch=len(hist))
    torch.save(best["state"], os.path.join(outdir, "model.pt"))
    json.dump(dict(cfg=cfg, hist=hist, best_epoch=best["epoch"], best_val_auc=best["auc"],
                   minutes=round((time.time() - t_start) / 60, 1)), open(os.path.join(outdir, "log.json"), "w"), indent=1)
    log(f"[{cfg['name']} s{cfg['seed']}] DONE best_val_auc={best['auc']:.4f} at epoch {best['epoch']} "
        f"({(time.time() - t_start) / 60:.1f} min)")
    return best["auc"]


def find_prep_dirs(files=None):
    """Locate NB00 output (dir with meta.csv + images.npy) and NB00b output (dir with meta.csv + masks.npy +
    dup_candidates.csv), independent of how Kaggle mounts notebook outputs. Prints a diagnosis if missing."""
    files = files if files is not None else walk_files("/kaggle/input") + walk_files("/kaggle/working")
    by_dir = {}
    for f in files:
        by_dir.setdefault(os.path.dirname(f), set()).add(os.path.basename(f))
    p1 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "images.npy"} <= fs)
    p2 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "masks.npy", "dup_candidates.csv"} <= fs)
    if not p1:
        print("!!! NB00 output NOT found under /kaggle/input. What IS mounted (first 3 levels):")
        for d in sorted({"/".join(f.split("/")[:6]) for f in files})[:40]:
            print("   ", d)
        print("   any meta.csv:", [f for f in files if f.endswith("meta.csv")][:10])
    return (p1[0] if p1 else None), (p2[0] if p2 else None)


In [ ]:
%%writefile evalkit.py
"""
evalkit.py -- evaluation toolkit for the paper (NB02).
Classification metrics + DeLong + bootstrap, convex last-layer refit, prototype attribution
(lesion / artifact / background hits), Grad-CAM for the CNN, controlled counterfactual
artifact interventions (lesion pixels provably unchanged) and explanation-stability metrics.
"""
import os, json, math, numpy as np, pandas as pd, cv2, torch
import torch.nn.functional as F
from scipy.stats import norm, wilcoxon
from sklearn.metrics import roc_auc_score
import artproto as ap

OUT = ap.OUT  # 224


# ----------------------------------------------------------------------------- runs
def find_runs(files):
    by = {}
    for f in files:
        by.setdefault(os.path.dirname(f), set()).add(os.path.basename(f))
    runs = {}
    for d, fs in by.items():
        name = os.path.basename(d)
        if {"model.pt", "log.json"} <= fs and not name.endswith("_smoke"):
            if name not in runs or d.startswith("/kaggle/working"):
                runs[name] = d
    return dict(sorted(runs.items()))


def load_run(d, device):
    L = json.load(open(f"{d}/log.json"))
    cfg = L["cfg"]
    sd = torch.load(f"{d}/model.pt", map_location="cpu")
    m = ap.build_model(cfg, pretrained=False)
    if "features.conv0.weight" in sd:          # real DenseNet-121 checkpoints (no weight download needed)
        import torchvision
        m.features = torchvision.models.densenet121(weights=None).features
    m.load_state_dict(sd, strict=True)
    m = m.to(device).eval().to(memory_format=torch.channels_last)
    return m, cfg, L


# ----------------------------------------------------------------------------- forward passes
def _batches(n, bs):
    for s in range(0, n, bs):
        yield s, min(n, s + bs)


@torch.no_grad()
def forward(model, imgs, msks, idx, device, bs=128, keep_maps=False):
    """imgs/msks: arrays or memmaps (N,256,256,3)/(N,256,256); idx: positions to use.
    Returns dict prob (n,), act (n,P) and optionally sim (n,P,7,7) float16."""
    model.eval()
    probs, acts, sims = [], [], []
    for s, e in _batches(len(idx), bs):
        ii = idx[s:e]
        x, _, _ = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(imgs[ii])).to(device),
                                 torch.from_numpy(np.ascontiguousarray(msks[ii])).to(device), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model(x.contiguous(memory_format=torch.channels_last))
        probs.append(F.softmax(out["logits"].float(), 1)[:, 1].cpu())
        if "act" in out:
            acts.append(out["act"].float().cpu())
            if keep_maps:
                sims.append(out["sim"].half().cpu())
    r = {"prob": torch.cat(probs).numpy()}
    if acts:
        r["act"] = torch.cat(acts).numpy()
        if keep_maps:
            r["sim"] = torch.cat(sims).numpy()
    return r


@torch.no_grad()
def gradcam(model, imgs, msks, idx, device, bs=128):
    """Class-activation map of the melanoma logit. For a global-average-pooling + linear head,
    d(logit_mel)/d(f) = w_mel/(h*w) at every position, so Grad-CAM equals CAM (Zhou et al. 2016)
    up to a positive constant; this formulation needs no backward pass (low memory)."""
    model.eval()
    w = model.head.weight[1].float()
    cams = []
    for s, e in _batches(len(idx), bs):
        ii = idx[s:e]
        x, _, _ = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(imgs[ii])).to(device),
                                 torch.from_numpy(np.ascontiguousarray(msks[ii])).to(device), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            f = F.relu(model.features(x.contiguous(memory_format=torch.channels_last)))
        cams.append(F.relu(torch.einsum("bchw,c->bhw", f.float(), w)).half().cpu())
    return torch.cat(cams).numpy()


# ----------------------------------------------------------------------------- convex last layer
def refit_last_layer(model, acts, y, class_w, l1=1e-4, steps=3000, lr=0.02):
    """Optimise the ProtoPNet last layer (linear map on prototype activations) to convergence:
    weighted CE + l1 on negative-class connections (as in Chen et al.). Deterministic, no val peeking."""
    dev = model.last.weight.device
    W = model.last.weight.detach().clone().float().requires_grad_(True)
    A = torch.as_tensor(acts, dtype=torch.float32, device=dev)
    Y = torch.as_tensor(y, dtype=torch.long, device=dev)
    cw = torch.as_tensor(class_w, dtype=torch.float32, device=dev)
    neg = (1 - model.class_identity).float()
    opt = torch.optim.Adam([W], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(steps):
        loss = F.cross_entropy(A @ W.T, Y, weight=cw) + l1 * (W * neg).abs().sum()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return W.detach(), float(loss.detach())


def probs_from_act(act, W):
    return torch.softmax(torch.as_tensor(act) @ torch.as_tensor(W).cpu().T, 1)[:, 1].numpy()


# ----------------------------------------------------------------------------- metrics
def youden_threshold(y, p):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, p)
    k = int(np.argmax(tpr - fpr))
    return float(min(thr[k], 1.0))


def bin_metrics(y, p, thr):
    y = np.asarray(y).astype(int); pr = (np.asarray(p) >= thr).astype(int)
    tp = int(((pr == 1) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
    fp = int(((pr == 1) & (y == 0)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); ppv = tp / max(tp + fp, 1)
    auc = roc_auc_score(y, p) if 0 < y.sum() < len(y) else float("nan")
    return dict(AUC=auc, BAcc=(sens + spec) / 2, Sens=sens, Spec=spec, PPV=ppv,
                F1=2 * ppv * sens / max(ppv + sens, 1e-9), TP=tp, FN=fn, FP=fp, TN=tn)


def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N); T2[J] = T + 1
    return T2


def delong(y, p1, p2):
    """Paired DeLong test (Sun & Xu 2014). Returns auc1, auc2, z, two-sided p."""
    y = np.asarray(y).astype(int)
    order = np.r_[np.where(y == 1)[0], np.where(y == 0)[0]]
    m = int(y.sum()); n = len(y) - m
    P = np.vstack([p1, p2])[:, order].astype(np.float64)
    tx = np.array([_midrank(r[:m]) for r in P]); ty = np.array([_midrank(r[m:]) for r in P])
    tz = np.array([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / math.sqrt(max(var, 1e-12))
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * norm.sf(abs(z)))


def bootstrap_auc_ci(y, p, groups=None, B=1000, seed=0):
    rs = np.random.RandomState(seed); y = np.asarray(y); p = np.asarray(p)
    if groups is None:
        groups = np.arange(len(y))
    ug, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == g)[0] for g in range(len(ug))]
    vals = []
    for _ in range(B):
        pick = rs.randint(0, len(ug), len(ug))
        ii = np.concatenate([members[g] for g in pick])
        if 0 < y[ii].sum() < len(ii):
            vals.append(roc_auc_score(y[ii], p[ii]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# ----------------------------------------------------------------------------- masks at network resolution
def masks224(msk_u8, device):
    """bit-field masks (b,256,256) -> dict of bool tensors (b,224,224) with the SAME resampling as the
    evaluation transform (bilinear, align_corners=False) and a 0.5 threshold."""
    m = torch.as_tensor(np.ascontiguousarray(msk_u8)).to(device).long()
    chans = {"L": (m & ap.BIT_L) > 0, "A": (m & ap.BIT_ART) > 0}
    for k, b in ap.ART_BITS.items():
        chans[k] = (m & b) > 0
    st = torch.stack([v.float() for v in chans.values()], 1)
    st = F.interpolate(st, size=(OUT, OUT), mode="bilinear", align_corners=False) >= 0.5
    return {k: st[:, i] for i, k in enumerate(chans)}


def hot_regions(maps7, size=OUT, q=0.95):
    """maps7 (b,k,7,7) float -> boolean top-(1-q) regions (b,k,size,size) after bilinear upsampling."""
    b, k = maps7.shape[:2]
    up = F.interpolate(maps7.float(), size=(size, size), mode="bilinear", align_corners=False)
    flat = up.reshape(b, k, -1)
    thr = torch.quantile(flat, q, dim=2, keepdim=True)
    return (flat >= thr).reshape(b, k, size, size), up


def region_stats(H, up, M):
    """H (b,k,S,S) bool hot regions; up the maps; M dict of (b,S,S) bool masks.
    Returns per-image means over the k maps: hit fraction of each mask + pointing (argmax in mask)."""
    hs = H.float().sum((2, 3)).clamp_min(1)
    arg = up.reshape(*up.shape[:2], -1).argmax(2)                     # (b,k)
    out = {}
    for name, m in M.items():
        mm = m[:, None].float()
        out[f"hit_{name}"] = ((H.float() * mm).sum((2, 3)) / hs).mean(1)
        out[f"point_{name}"] = torch.gather(m.reshape(m.shape[0], -1).float(), 1, arg).mean(1)
    return {k: v.cpu().numpy() for k, v in out.items()}


def attribution(maps7, acts, msk_u8, device, topk=3, q=0.95, bs=64):
    """Per-image attribution statistics for prototype models (top-k most activated prototypes) or the
    CNN (acts=None: maps7 is the (n,7,7) Grad-CAM)."""
    rows = {}
    n = len(maps7)
    for s, e in _batches(n, bs):
        mp = torch.as_tensor(maps7[s:e]).to(device).float()
        if acts is None:
            sel = mp[:, None]
        else:
            top = torch.as_tensor(np.argsort(-acts[s:e], 1)[:, :topk].copy()).to(device)
            sel = torch.gather(mp, 1, top[:, :, None, None].expand(-1, -1, mp.shape[2], mp.shape[3]))
        H, up = hot_regions(sel, q=q)
        M = masks224(msk_u8[s:e], device)
        M["B"] = ~M["L"] & ~M["A"]
        st = region_stats(H, up, M)
        for k, v in st.items():
            rows.setdefault(k, []).append(v)
        for k in ["L", "A"] + list(ap.ART_BITS):
            rows.setdefault(f"area_{k}", []).append(M[k].float().mean((1, 2)).cpu().numpy())
    return pd.DataFrame({k: np.concatenate(v) for k, v in rows.items()})


# ----------------------------------------------------------------------------- counterfactual interventions
def _allowed(L, margin=4):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1))
    return cv2.dilate(L.astype(np.uint8), k) == 0


def _compose(img, layer, allowed):
    changed = (layer != img).any(-1) & allowed
    out = img.copy(); out[changed] = layer[changed]
    return out, changed


def add_hair(img, L, rng):
    layer = img.copy()
    for _ in range(int(rng.integers(12, 30))):
        p0 = rng.uniform(0, 256, 2); a = rng.uniform(0, 2 * np.pi); ln = rng.uniform(60, 200)
        p2 = p0 + ln * np.array([np.cos(a), np.sin(a)]); p1 = (p0 + p2) / 2 + rng.normal(0, 25, 2)
        t = np.linspace(0, 1, 60)[:, None]
        pts = ((1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2).astype(np.int32)
        c = rng.uniform(15, 70)
        cv2.polylines(layer, [pts], False, (int(c * 1.15), int(c * 0.95), int(c * 0.8)),
                      int(rng.integers(1, 3)), cv2.LINE_AA)
    return _compose(img, layer, _allowed(L))


def add_ruler(img, L, rng):
    allowed = _allowed(L)
    for side in rng.permutation(["top", "bottom", "left", "right"]):
        layer = img.copy(); col = int(rng.uniform(35, 80)); ln = int(rng.uniform(150, 220))
        off = int(rng.uniform(10, 30)); s0 = (256 - ln) // 2
        canvas = np.zeros((256, 256), np.uint8)
        cv2.line(canvas, (s0, off), (s0 + ln, off), 255, 2, cv2.LINE_AA)
        for k, x in enumerate(range(s0, s0 + ln + 1, 7)):
            cv2.line(canvas, (x, off), (x, off + (12 if k % 5 == 0 else 6)), 255, 1, cv2.LINE_AA)
        if side == "bottom": canvas = canvas[::-1]
        elif side == "left": canvas = canvas.T.copy()
        elif side == "right": canvas = canvas.T[:, ::-1].copy()
        drawn = canvas > 0
        if (drawn & ~allowed).sum() > 0.02 * drawn.sum():
            continue                                    # would touch the lesion margin -> try another side
        a = (canvas.astype(np.float32) / 255)[..., None]
        layer = (img * (1 - a) + col * a).astype(np.uint8)
        return _compose(img, layer, allowed)
    return None, None


def add_dark_corners(img, L, rng):
    yy, xx = np.mgrid[:256, :256]
    r = np.hypot(xx - 127.5, yy - 127.5) / (127.5 * np.sqrt(2))
    r0 = rng.uniform(0.72, 0.85)
    a = np.clip((r - r0) / 0.06, 0, 1)[..., None]
    layer = (img * (1 - a) + rng.uniform(0, 15) * a).astype(np.uint8)
    return _compose(img, layer, _allowed(L))


def add_ink(img, L, rng):
    ys, xs = np.where(L > 0)
    if len(ys) == 0:
        return None, None
    cy, cx = ys.mean(), xs.mean()
    ry = (ys.max() - ys.min()) / 2 + rng.uniform(12, 25); rx = (xs.max() - xs.min()) / 2 + rng.uniform(12, 25)
    layer = img.copy(); col = (int(rng.uniform(100, 135)), int(rng.uniform(30, 55)), int(rng.uniform(130, 170)))
    start = rng.uniform(0, 360)
    for k in range(int(rng.integers(2, 5))):
        a0 = start + k * 90 + rng.uniform(-10, 10)
        cv2.ellipse(layer, (int(cx), int(cy)), (int(rx), int(ry)), 0, a0, a0 + rng.uniform(35, 70), col, 3, cv2.LINE_AA)
    return _compose(img, layer, _allowed(L))


def remove_by_inpaint(img, L, mask):
    m = (cv2.dilate(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & _allowed(L, 2)
    if m.sum() == 0:
        return None, None
    inp = cv2.inpaint(img, m.astype(np.uint8), 3, cv2.INPAINT_TELEA)
    out = img.copy(); out[m] = inp[m]
    return out, m


def null_inpaint(img, L, art_mask, rng):
    """Control: inpaint a region of the same shape as the artifact mask, shifted onto clean skin."""
    for _ in range(10):
        dy, dx = rng.integers(-100, 100, 2)
        sh = np.roll(np.roll(art_mask, dy, 0), dx, 1) & ~art_mask
        if sh.sum() >= 0.5 * art_mask.sum():
            return remove_by_inpaint(img, L, sh)
    return None, None


INTERVENTIONS = ["add_hair", "add_ruler", "add_dark_corners", "add_ink", "remove_hair_ruler", "null_inpaint"]


def make_counterfactuals(IM, MS, idx, kind, seed=0):
    """Returns (cf images (n,256,256,3) uint8, changed masks (n,256,256) bool, ok flags) for test indices idx.
    Lesion pixels are verified to be bit-identical to the original."""
    n = len(idx)
    CF = np.zeros((n, 256, 256, 3), np.uint8); CH = np.zeros((n, 256, 256), bool); ok = np.zeros(n, bool)
    for k, i in enumerate(idx):
        img = np.array(IM[i]); m = np.array(MS[i]); L = (m & ap.BIT_L) > 0
        rng = np.random.default_rng([seed, int(i), INTERVENTIONS.index(kind)])
        if kind == "add_hair": out, ch = add_hair(img, L, rng)
        elif kind == "add_ruler": out, ch = add_ruler(img, L, rng)
        elif kind == "add_dark_corners": out, ch = add_dark_corners(img, L, rng)
        elif kind == "add_ink": out, ch = add_ink(img, L, rng)
        elif kind == "remove_hair_ruler":
            ts = (m & ap.BIT_TS) > 0
            out, ch = remove_by_inpaint(img, L, ts) if ts.mean() >= ap.ART_MIN_FRAC["TS"] else (None, None)
        elif kind == "null_inpaint":
            ts = (m & ap.BIT_TS) > 0
            out, ch = null_inpaint(img, L, ts | False, rng) if ts.mean() >= ap.ART_MIN_FRAC["TS"] else (None, None)
            if out is not None:
                ch = ch & ((m & ap.BIT_ART) == 0)
                out2 = img.copy(); out2[ch] = out[ch]; out = out2
        else:
            raise ValueError(kind)
        if out is None or ch.sum() == 0:
            CF[k] = img; continue
        assert np.array_equal(out[L], img[L]), f"lesion pixels changed for {kind} image {i}"
        CF[k] = out; CH[k] = ch; ok[k] = True
    return CF, CH, ok


def stability(sim0, act0, sim1, act1, changed256, msk_u8, device, topk=3, q=0.95, bs=64):
    """Explanation stability for prototype models: same top-k prototypes (chosen on the original image)
    before vs after the intervention."""
    rows = {}
    for s, e in _batches(len(sim0), bs):
        a0 = act0[s:e]; top = torch.as_tensor(np.argsort(-a0, 1)[:, :topk].copy()).to(device)
        g = lambda S: torch.gather(torch.as_tensor(S[s:e]).to(device).float(), 1,
                                   top[:, :, None, None].expand(-1, -1, S.shape[2], S.shape[3]))
        m0, m1 = g(sim0), g(sim1)
        H0, _ = hot_regions(m0, q=q); H1, _ = hot_regions(m1, q=q)
        M = masks224(msk_u8[s:e], device)
        ch = F.interpolate(torch.as_tensor(changed256[s:e]).to(device).float()[:, None], size=(OUT, OUT),
                           mode="bilinear", align_corners=False)[:, 0] >= 0.5
        inter = (H0 & H1).float().sum((2, 3)); uni = (H0 | H1).float().sum((2, 3)).clamp_min(1)
        L = M["L"][:, None].float()
        h1 = H1.float().sum((2, 3)).clamp_min(1); h0 = H0.float().sum((2, 3)).clamp_min(1)
        rows.setdefault("iou", []).append((inter / uni).mean(1).cpu().numpy())
        rows.setdefault("les_before", []).append(((H0.float() * L).sum((2, 3)) / h0).mean(1).cpu().numpy())
        rows.setdefault("les_after", []).append(((H1.float() * L).sum((2, 3)) / h1).mean(1).cpu().numpy())
        rows.setdefault("capture", []).append(((H1.float() * ch[:, None].float()).sum((2, 3)) / h1).mean(1).cpu().numpy())
        t1 = np.argsort(-act1[s:e], 1)[:, :topk]; t0 = np.argsort(-a0, 1)[:, :topk]
        rows.setdefault("top_jaccard", []).append(np.array([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(t0, t1)]))
        x0 = m0[:, 0].reshape(m0.shape[0], -1); x1 = m1[:, 0].reshape(m1.shape[0], -1)
        r0 = x0.argsort(1).argsort(1).float(); r1 = x1.argsort(1).argsort(1).float()
        r0 = r0 - r0.mean(1, keepdim=True); r1 = r1 - r1.mean(1, keepdim=True)
        rows.setdefault("spearman_top1", []).append(((r0 * r1).sum(1) / (r0.norm(dim=1) * r1.norm(dim=1)).clamp_min(1e-9)).cpu().numpy())
    return pd.DataFrame({k: np.concatenate(v) for k, v in rows.items()})


def paired_wilcoxon(a, b):
    d = np.asarray(a) - np.asarray(b)
    d = d[np.isfinite(d)]
    if len(d) < 10 or np.allclose(d, 0):
        return float("nan")
    return float(wilcoxon(d).pvalue)


## 0. Software environment (for the pinned requirements of the repository)

In [ ]:
import sys, platform, importlib, subprocess
print('python', sys.version.split()[0], '|', platform.platform())
for m in ('torch', 'torchvision', 'numpy', 'scipy', 'sklearn', 'pandas', 'cv2', 'statsmodels', 'PIL', 'matplotlib', 'openpyxl'):
    try: print(f'{m:12s}', importlib.import_module(m).__version__)
    except Exception as e: print(f'{m:12s} not available ({e.__class__.__name__})')
try:
    import torch; print('cuda', torch.version.cuda, '| cudnn', torch.backends.cudnn.version())
except Exception: pass

## 1. Setup

In [ ]:
import os, time, warnings, numpy as np, pandas as pd, torch, cv2
import torch.nn.functional as F
import importlib, artproto as ap, evalkit as ek; importlib.reload(ap); importlib.reload(ek)
from scipy.stats import rankdata, mannwhitneyu
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 500)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
ROOT = os.environ.get('NB11_ROOT', '/kaggle'); SMOKE = int(os.environ.get('NB11_SMOKE', '0'))
T0 = time.time()
B_BOOT = 200 if SMOKE else 2000
FILES = ap.walk_files(f'{ROOT}/input') + ap.walk_files(f'{ROOT}/working')
P1, P2 = ap.find_prep_dirs(FILES); PM = P2 or P1
RUNS = ek.find_runs(FILES)
cfg_of = lambda r: r.rsplit('_s', 1)[0]
COUNT = pd.Series([cfg_of(r) for r in RUNS]).value_counts().to_dict()
print('runs per configuration:', COUNT)
MAIN = ['CNN', 'PPN-std', 'PPN', 'PPN-FA', 'Ours', 'Ours-noLes', 'Ours-noArt', 'Ours-stdPush']
MASKLOSS = ['PPN-FA-Correia', 'PPN-FA-IAIABL']
INDEP = ['PPN-indep', 'Correia-indep', 'Ours-indep']
for c in MAIN + MASKLOSS: assert COUNT.get(c, 0) == 3, f'{c}: {COUNT.get(c, 0)} runs'
for c in INDEP: assert COUNT.get(c, 0) == 4, f'{c}: {COUNT.get(c, 0)} runs'
FAMILY = {'PPN': ['PPN', 'PPN-indep'], 'Correia': ['PPN-FA-Correia', 'Correia-indep'], 'Ours': ['Ours', 'Ours-indep']}
EVAL = [r for r in RUNS if cfg_of(r) in MAIN + MASKLOSS + INDEP]
OUT = f'{ROOT}/working/nb13'; os.makedirs(OUT, exist_ok=True)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
meta = pd.read_csv(f'{PM}/meta.csv')
IM = np.load(f'{P1}/images.npy', mmap_mode='r'); MS = np.load(f'{PM}/masks.npy', mmap_mode='r')
assert len(meta) == IM.shape[0] == MS.shape[0]
y = meta.label.values.astype(int)
IDX = {s: np.where(meta.split.values == s)[0] for s in ('train', 'val', 'test')}
if not SMOKE:
    assert {k: len(v) for k, v in IDX.items()} == {'train': 20552, 'val': 2459, 'test': 2320}, 'split differs from the paper'
TR, VA, TEST = IDX['train'], IDX['val'], IDX['test']
n1 = y[TR].sum(); n0 = len(TR) - n1
CLASS_W = [len(TR) / (2 * n0), len(TR) / (2 * n1)]
GROUPS = meta.group.values[TEST]
VALID = meta.lesion_valid.values.astype(bool)
ytr, yte = y[TR], y[TEST]
keep = VALID[TEST]; CF_IDX = TEST[keep]
pos_of = {g: k for k, g in enumerate(TEST)}; loc = np.array([pos_of[g] for g in CF_IDX])
print('data:', P1, '| runs evaluated:', len(EVAL), '| device:', dev, flush=True)

In [ ]:
# Derm7pt: dermoscopic images (prepared as in the paper) and the seven-point criteria of the original metadata
D7 = None
em = pd.read_csv(f'{PM}/ext_Derm7pt-derm_meta.csv')
D7_IM = np.load(f'{P1}/ext_Derm7pt-derm_images.npy', mmap_mode='r'); D7_MS = np.load(f'{PM}/ext_Derm7pt-derm_masks.npy', mmap_mode='r')
CRIT = ['pigment_network', 'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules', 'blue_whitish_veil', 'vascular_structures']
for f in [f for f in FILES if os.path.basename(f) == 'meta.csv']:
    try: m = pd.read_csv(f)
    except Exception: continue
    if {'diagnosis', 'derm', 'clinic'} <= set(m.columns) and set(CRIT) <= set(m.columns):
        D7 = em[['image', 'label']].merge(m[['derm'] + CRIT], left_on='image', right_on='derm', how='left'); break
assert D7 is not None and D7[CRIT].notna().all().all() and len(D7) == len(em), 'Derm7pt criteria not found / not aligned'
for c in CRIT:
    print(f'{c:22s}', D7[c].value_counts().to_dict())
    D7[c] = (D7[c].astype(str).str.strip().str.lower() != 'absent').astype(int)
    assert 0 < D7[c].sum() < len(D7), f'{c}: no "absent" category found'
print('\nprevalence of each criterion (present) by diagnosis:')
print(D7.groupby('label')[CRIT].agg(['sum', 'count']).T.unstack().to_string())

In [ ]:
import io, re, zipfile, base64, urllib.request
from PIL import Image
from scipy.stats import spearmanr, wilcoxon

def auc_fast(yb, p):
    yb = np.asarray(yb).astype(int); r = rankdata(p); n_1 = yb.sum(); n_0 = len(yb) - n_1
    return (r[yb == 1].sum() - n_1 * (n_1 + 1) / 2) / (n_1 * n_0) if n_1 and n_0 else np.nan

def mw(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if not (len(a) and len(b)): return np.nan
    method = 'exact' if len(set(a) | set(b)) == len(a) + len(b) else 'asymptotic'
    return float(mannwhitneyu(a, b, alternative='two-sided', method=method).pvalue)

def boot_groups(fn, groups, B=B_BOOT, seed=0):
    """paired cluster bootstrap over lesion groups of a scalar statistic fn(index array): 95% CI and two-sided p"""
    rs = np.random.RandomState(seed); u, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(u))]; d = []
    for _ in range(B):
        ii = np.concatenate([members[k] for k in rs.randint(0, len(u), len(u))])
        v = fn(ii)
        if np.isfinite(v): d.append(v)
    d = np.array(d); lo, hi = np.percentile(d, [2.5, 97.5])
    return float(lo), float(hi), float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))

def layers(IMa, MSa, idx, dil=7):
    """background only (dilated lesion inpainted, Telea) and a blending weight that is exactly 1 on the lesion and falls
    linearly to 0 across the 7-px dilation ring (pasted lesion pixels stay bit-identical)"""
    n = len(idx); BG = np.zeros((n, 256, 256, 3), np.uint8); AL = np.zeros((n, 256, 256), np.float32); area = np.zeros(n)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    for t, i in enumerate(idx):
        img = np.array(IMa[i]); L = ((np.array(MSa[i]) & ap.BIT_L) > 0).astype(np.uint8)
        R = cv2.dilate(L, k1); area[t] = R.mean()
        BG[t] = cv2.inpaint(img, R, 5, cv2.INPAINT_TELEA)
        dist = cv2.distanceTransform((1 - L).astype(np.uint8), cv2.DIST_L2, 5)
        a = np.clip(1.0 - dist / (dil + 1), 0, 1).astype(np.float32); a[L > 0] = 1.0; a[R == 0] = 0.0
        AL[t] = a
    return BG, AL, area

def composite(IMa, idx, AL, BG, partner):
    out = np.zeros((len(idx), 256, 256, 3), np.uint8)
    for t, i in enumerate(idx):
        img = np.array(IMa[i]).astype(np.float32); a = AL[t][..., None]
        c = np.rint(a * img + (1 - a) * BG[partner[t]].astype(np.float32)).astype(np.uint8)
        les = AL[t] >= 1.0
        assert np.array_equal(c[les], np.array(IMa[i])[les])
        out[t] = c
    return out

def participation_ratio(act):
    sd = act.std(0); live = sd > 1e-6 * max(sd.max(), 1e-12)
    R = np.corrcoef(rankdata(act[:, live], axis=0), rowvar=False) if live.sum() > 1 else np.ones((1, 1))
    R = np.nan_to_num(np.atleast_2d(R), nan=0.0); np.fill_diagonal(R, 1.0)
    ev = np.clip(np.linalg.eigvalsh(R), 0, None); off = np.abs(R[~np.eye(len(R), dtype=bool)])
    return float(ev.sum() ** 2 / (ev ** 2).sum()), float(off.mean()) if off.size else 1.0

## 1. Edited test sets, expert attribute masks, diagnoses, external sets

In [ ]:
# (A) background swap: partners drawn once (seed 0) from other lesion groups; identical for every network
labs = y[CF_IDX]; grp = meta.group.values[CF_IDX]; n = len(CF_IDX)
BG, AL, AREA = layers(IM, MS, CF_IDX); MS_K = np.array(MS[CF_IDX]); ar = np.arange(n)
rs = np.random.RandomState(0)
def pick(t, pool):
    for _ in range(1000):
        j = pool[rs.randint(len(pool))]
        if grp[j] != grp[t]: return j
    raise RuntimeError('no partner')
POS_MEL, POS_BEN = np.where(labs == 1)[0], np.where(labs == 0)[0]
P_MEL = np.array([pick(t, POS_MEL) for t in range(n)]); P_BEN = np.array([pick(t, POS_BEN) for t in range(n)])
P_RAND = np.array([pick(t, np.arange(n)) for t in range(n)])
C_MEL, C_BEN, C_RAND = (composite(IM, CF_IDX, AL, BG, p) for p in (P_MEL, P_BEN, P_RAND))
print(f'background swap: {n} test lesions | melanoma {len(POS_MEL)} | random partners of the same class: {(labs[P_RAND] == labs).mean():.2f} | '
      f'lesion-area AUC for melanoma (size cue left in the background-only images): {auc_fast(labs, AREA):.3f}', flush=True)

# (B) expert dermoscopic attributes of ISIC 2018 Task 2
ATTR = ['pigment_network', 'negative_network', 'streaks', 'milia_like_cyst', 'globules']
ids = (meta.image.astype(str).str[:12].values if 'image' in meta.columns else np.array([f'ISIC_{i:07d}' for i in range(len(meta))]))
AMAP = {}
if not SMOKE:
    for tag, url in [('train', 'https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task2_Training_GroundTruth_v3.zip'),
                     ('val', 'https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task2_Validation_GroundTruth.zip')]:
        dst = f'{ROOT}/working/isic2018_task2_{tag}.zip'
        try:
            if not os.path.exists(dst): urllib.request.urlretrieve(url, dst)
            z = zipfile.ZipFile(dst)
            for nm in z.namelist():
                m = re.search(r'(ISIC_\d{7})_attribute_([a-z_]+)\.png$', nm)
                if m: AMAP.setdefault(m.group(1), {})[m.group(2)] = (z, nm)
            print(f'ISIC 2018 Task 2 {tag}: {os.path.getsize(dst) / 1e6:.1f} MB', flush=True)
        except Exception as e:
            print(f'!!! ISIC 2018 Task 2 {tag} not available: {e.__class__.__name__}: {e}', flush=True)
def load_attr(i):
    out = np.zeros((len(ATTR), 256, 256), np.uint8)
    for k, a in enumerate(ATTR):
        z, nm = AMAP[ids[i]][a]
        m = np.asarray(Image.open(io.BytesIO(z.read(nm))).convert('L'))
        out[k] = (cv2.resize(m, (256, 256), interpolation=cv2.INTER_AREA) >= 128)
    return out
matched = np.array([i for i in range(len(meta)) if ids[i] in AMAP and all(a in AMAP[ids[i]] for a in ATTR)], int)
if SMOKE:   # synthetic structures inside the lesion to exercise the code
    matched = np.where(meta.lesion_valid.values.astype(bool))[0]
    def load_attr(i):
        L = (np.array(MS[i]) & ap.BIT_L) > 0; yy, xx = np.mgrid[:256, :256]
        return np.stack([(L & ((xx // (8 + 4 * k) + yy // 16) % 2 == 0)).astype(np.uint8) for k in range(len(ATTR))])
SPL = meta.split.values
AT_EVAL = matched[np.isin(SPL[matched], ['val', 'test']) & VALID[matched]]
AT_TRAIN = matched[(SPL[matched] == 'train') & VALID[matched]]
AT = {k: np.stack([load_attr(i) for i in v]) if len(v) else np.zeros((0, len(ATTR), 256, 256), np.uint8) for k, v in [('eval', AT_EVAL), ('train', AT_TRAIN)]}
print('ISIC 2018 Task 2 images in our data:', len(matched), '| validation+test:', len(AT_EVAL), '| training:', len(AT_TRAIN),
      '| sources:', meta.source.values[matched].tolist().count('MSK') if 'source' in meta.columns else 'n/a')
if len(AT_EVAL):
    print('attribute present (validation+test):', {a: int(AT['eval'][:, k].reshape(len(AT_EVAL), -1).any(1).sum()) for k, a in enumerate(ATTR)})

# (C) diagnoses of the benign test images
DX = ['NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
if all(d in meta.columns for d in DX):
    DXM = meta[DX].values.astype(int)
else:
    assert SMOKE, 'diagnosis columns missing in meta.csv'
    DXM = np.zeros((len(meta), len(DX)), int); r0 = np.random.RandomState(1)
    DXM[np.arange(len(meta)), r0.randint(0, len(DX), len(meta))] = 1; DXM[y == 1] = 0
BEN_TE = np.where(yte == 0)[0]
print('benign test images per diagnosis:', dict(zip(DX, DXM[TEST][BEN_TE].sum(0).tolist())))

# external sets
EXT = {}
for name in ('Derm7pt-derm', 'Derm7pt-clinic', 'PAD-UFES-20'):
    f = f'{PM}/ext_{name}_meta.csv'
    if not os.path.exists(f): print('missing external set', name); continue
    em = pd.read_csv(f); k = np.where(em.label.values >= 0)[0]
    EXT[name] = dict(y=em.label.values[k].astype(int), k=k, IM=np.load(f'{P1}/ext_{name}_images.npy', mmap_mode='r'),
                     MS=np.load(f'{PM}/ext_{name}_masks.npy', mmap_mode='r'))
print('external sets:', {k: len(v['k']) for k, v in EXT.items()}, flush=True)

## 2. Every network

In [ ]:
@torch.no_grad()
def attr_stats(model, proto, idx, ATm, bs=32):
    if not len(idx): return pd.DataFrame()
    if proto:
        o = ek.forward(model, IM, MS, idx, dev, keep_maps=True); maps, acts = o['sim'], o['act']
    else:
        maps, acts = ek.gradcam(model, IM, MS, idx, dev), None
    rows = []
    for s, e in ek._batches(len(idx), bs):
        mp = torch.as_tensor(maps[s:e]).to(dev).float()
        if acts is None: sel = mp[:, None]
        else:
            top = torch.as_tensor(np.argsort(-acts[s:e], 1)[:, :3].copy()).to(dev)
            sel = torch.gather(mp, 1, top[:, :, None, None].expand(-1, -1, mp.shape[2], mp.shape[3]))
        H, up = ek.hot_regions(sel); Hf = H.float(); hs = Hf.sum((2, 3)).clamp_min(1)
        A5 = F.interpolate(torch.as_tensor(ATm[s:e]).to(dev).float(), size=(ek.OUT, ek.OUT), mode='bilinear', align_corners=False) >= 0.5
        A = A5.any(1); Lm = ek.masks224(np.array(MS[idx[s:e]]), dev)['L']; AL_ = A & Lm
        hL = (Hf * Lm[:, None].float()).sum((2, 3))
        share = torch.where(hL > 0, (Hf * AL_[:, None].float()).sum((2, 3)) / hL.clamp_min(1), torch.full_like(hL, float('nan')))
        chance = AL_.float().sum((1, 2)) / Lm.float().sum((1, 2)).clamp_min(1)
        arg = up.reshape(*up.shape[:2], -1).argmax(2)[:, 0]
        point = torch.gather(A.reshape(A.shape[0], -1).float(), 1, arg[:, None])[:, 0]
        r = dict(hit_attr=((Hf * A[:, None].float()).sum((2, 3)) / hs).mean(1), hit_lesion=(hL / hs).mean(1),
                 share_in=torch.nanmean(share, 1), chance=chance, point_attr=point, attr_in_lesion=AL_.float().sum((1, 2)))
        for k, a in enumerate(ATTR):
            r[f'hit {a}'] = ((Hf * A5[:, k][:, None].float()).sum((2, 3)) / hs).mean(1)
            r[f'area {a}'] = A5[:, k].float().mean((1, 2))
        rows.append({k: v.cpu().numpy() for k, v in r.items()})
    D = pd.DataFrame({k: np.concatenate([r[k] for r in rows]) for k in rows[0]})
    ok = D.attr_in_lesion.values > 0
    D['excess'] = np.where(ok, D.share_in - D.chance, np.nan); D.loc[~ok, 'point_attr'] = np.nan
    return D

def dx_cover(ate, cls):
    A = ate[BEN_TE][:, cls == 0]; Y = DXM[TEST][BEN_TE]
    M = np.array([[auc_fast(Y[:, d], A[:, j]) for j in range(A.shape[1])] for d in range(len(DX))])    # (dx, benign prototypes)
    best = np.nanmax(M, 1); pmax = np.nanmax(M, 0); parg = np.nanargmax(np.nan_to_num(M, nan=-1), 0)
    out = dict(dx_best_mean=float(np.nanmean(best)), dx_covered=int((best >= 0.75).sum()),
               dx_specialists=int(len(set(parg[pmax >= 0.70]))))
    out.update({f'dx {d}': float(b) for d, b in zip(DX, best)})
    return out

ROWS, PRED, ATT, GEO = [], {}, {}, []
for run in EVAL:
    t = time.time(); model, cfg, L = ek.load_run(RUNS[run], dev); proto = cfg['model'] == 'proto'
    if proto:
        A_tr = ek.forward(model, IM, MS, TR, dev)['act']
        W, _ = ek.refit_last_layer(model, A_tr, ytr, CLASS_W); W = W.cpu().numpy(); del A_tr
        f = lambda imgs, msks, idx: (lambda o: (ek.probs_from_act(o['act'], W), o['act']))(ek.forward(model, imgs, msks, idx, dev))
    else:
        f = lambda imgs, msks, idx: (ek.forward(model, imgs, msks, idx, dev)['prob'], None)
    pva, _ = f(IM, MS, VA); thr = ek.youden_threshold(y[VA], pva)
    pte, ate = f(IM, MS, TEST)
    rec = dict(run=run, config=cfg_of(run), auc=roc_auc_score(yte, pte))
    for name, E in EXT.items():
        rec[f'auc {name}'] = roc_auc_score(E['y'], f(E['IM'], E['MS'], E['k'])[0])
    p_bg, _ = f(BG, MS_K, ar)
    rec.update(auc_bg=auc_fast(labs, p_bg), rho_bg_area=float(spearmanr(p_bg, AREA).statistic))
    pm, _ = f(C_MEL, MS_K, ar); pb, _ = f(C_BEN, MS_K, ar); pr, _ = f(C_RAND, MS_K, ar)
    ps = np.where(labs == 1, pm, pb); po = np.where(labs == 1, pb, pm)
    rec.update(auc_orig=auc_fast(labs, pte[loc]), auc_same=auc_fast(labs, ps), auc_rand=auc_fast(labs, pr), auc_opp=auc_fast(labs, po))
    rec.update(bg_gap=rec['auc_same'] - rec['auc_rand'], bg_gap_opp=rec['auc_same'] - rec['auc_opp'],
               dp_bgclass=float(np.mean(pm - pb)), flip_bgclass=float(np.mean((pm >= thr) != (pb >= thr))))
    PRED[run] = dict(pm=pm, pb=pb, pr=pr, p=pte[loc])
    for split, idx in [('eval', AT_EVAL), ('train', AT_TRAIN)]:
        D = attr_stats(model, proto, idx, AT[split])
        if len(D):
            ATT[(run, split)] = D
            rec.update({f'{split} {k}': float(np.nanmean(D[k])) for k in ['excess', 'point_attr', 'hit_attr', 'hit_lesion', 'share_in', 'chance']})
    if proto:
        cls = model.proto_class.cpu().numpy(); rec.update(dx_cover(ate, cls))
        rec['pr_all'], rec['abs_r'] = participation_ratio(ate)
        pf = f'{RUNS[run]}/push_info.csv'
        if os.path.exists(pf):
            info = pd.read_csv(pf); hh, ww = info.h.values, info.w.values
            ring = (hh == 0) | (hh == 6) | (ww == 0) | (ww == 6); corner = np.isin(hh, [0, 6]) & np.isin(ww, [0, 6])
            rec.update(ring_protos=int(ring.sum()), corner_protos=int(corner.sum()), lesion_protos=int((info.les_frac >= 0.5).sum()))
            for j in range(len(info)):
                GEO.append(dict(run=run, config=cfg_of(run), proto=j, cls=int(info.cls[j]), img=int(info.img[j]), h=int(hh[j]), w=int(ww[j]),
                                les_frac=float(info.les_frac[j]), art_frac=float(info.art_frac[j]), ring=bool(ring[j])))
    ROWS.append(rec); del model; torch.cuda.empty_cache()
    print(f"{run:20s} AUC {rec['auc']:.4f} | BG-Gap {rec['bg_gap']:+.3f} | dp(bg class) {rec['dp_bgclass']:+.3f} | "
          f"excess {rec.get('eval excess', np.nan):+.3f} | dx cover {rec.get('dx_best_mean', np.nan):.3f} | "
          f"ring {rec.get('ring_protos', '-')} | {time.time() - t:.0f}s", flush=True)
R = pd.DataFrame(ROWS); R.to_csv(f'{OUT}/per_run.csv', index=False); pd.DataFrame(GEO).to_csv(f'{OUT}/prototype_sources.csv', index=False)

## 3. Results: per configuration, across seven runs, seed ensembles

In [ ]:
FAM = {'PPN-indep': 'PPN', 'Correia-indep': 'PPN-FA-Correia', 'Ours-indep': 'Ours'}
R['fam'] = R.config.replace(FAM)
ORDER = ['CNN', 'PPN-std', 'PPN', 'PPN-FA', 'Ours', 'Ours-noLes', 'Ours-noArt', 'Ours-stdPush', 'PPN-FA-Correia', 'PPN-FA-IAIABL']
MET = (['auc', 'auc_orig', 'auc_same', 'auc_rand', 'auc_opp', 'bg_gap', 'bg_gap_opp', 'dp_bgclass', 'flip_bgclass', 'auc_bg', 'rho_bg_area']
       + [c for c in R.columns if c.startswith('auc ') or c.startswith('eval ') or c.startswith('train ') or c.startswith('dx')]
       + ['pr_all', 'abs_r', 'ring_protos', 'corner_protos', 'lesion_protos'])
MET = [m for m in MET if m in R.columns]
TAB = R.groupby('fam')[MET].agg(lambda v: f'{np.nanmean(v):.3f}±{np.nanstd(v):.3f}').T
print('Mean ± std over runs (Ours, PPN, Correia: seven runs; others: three):')
print(TAB[[c for c in ORDER if c in TAB.columns]].to_string())
print('\nper run:'); print(R[['run'] + MET].round(4).to_string(index=False))

POOL = ['PPN-std', 'PPN', 'PPN-FA']
TESTS = [('bg_gap', 'lower'), ('dp_bgclass', 'lower'), ('flip_bgclass', 'lower'), ('eval excess', 'higher'), ('eval point_attr', 'higher'),
         ('train excess', 'higher'), ('dx_best_mean', 'higher'), ('dx_covered', 'higher'), ('dx_specialists', 'higher'),
         ('auc Derm7pt-derm', 'higher'), ('auc Derm7pt-clinic', 'higher'), ('auc PAD-UFES-20', 'higher'), ('auc_bg', 'lower'), ('pr_all', 'higher')]
rows = []
for met, better in TESTS:
    if met not in R.columns: continue
    a = R[R.fam == 'Ours'][met].values
    for other in ['PPN-std', 'PPN', 'PPN-FA', 'PPN-FA-Correia', 'PPN-FA-IAIABL', 'CNN', 'Ours-noLes', 'Ours-stdPush', 'POOL13']:
        b = R[R.fam.isin(POOL) if other == 'POOL13' else R.fam == other][met].values.astype(float)
        if not len(b) or np.isnan(b).all(): continue
        sep = bool(np.nanmax(a) < np.nanmin(b)) if better == 'lower' else bool(np.nanmin(a) > np.nanmax(b))
        rows.append(dict(metric=met, better=better, versus=other, ours=np.nanmean(a), other=np.nanmean(b), n=(len(a), len(b)), separated=sep, p_runs=mw(a, b)))
RUNTEST = pd.DataFrame(rows); RUNTEST.to_csv(f'{OUT}/tests_across_runs.csv', index=False)
print('\nAcross training runs (exact Mann-Whitney, two-sided):'); print(RUNTEST.round(4).to_string(index=False))

# seed ensembles (seeds 0-2): background swap with a paired bootstrap over lesion groups
def ens(cfg, key):
    rr = [r for r in PRED if cfg_of(r) == cfg]
    return np.mean([PRED[r][key] for r in rr], 0) if rr else None
ENSD = {}
for c in ORDER:
    if ens(c, 'pm') is None: continue
    pm, pb, pr = ens(c, 'pm'), ens(c, 'pb'), ens(c, 'pr'); ENSD[c] = (np.where(labs == 1, pm, pb), pr, pm - pb)
def gap(c, ii): s, r, _ = ENSD[c]; return auc_fast(labs[ii], s[ii]) - auc_fast(labs[ii], r[ii])
rows = []
for c in ENSD:
    if c == 'Ours': continue
    lo, hi, p = boot_groups(lambda ii: gap('Ours', ii) - gap(c, ii), grp)
    lo2, hi2, p2 = boot_groups(lambda ii: np.mean(ENSD['Ours'][2][ii]) - np.mean(ENSD[c][2][ii]), grp)
    rows.append(dict(versus=c, gap_ours=gap('Ours', ar), gap_other=gap(c, ar), diff_CI=f'[{lo:+.3f}, {hi:+.3f}]', p=p,
                     dp_ours=float(np.mean(ENSD['Ours'][2])), dp_other=float(np.mean(ENSD[c][2])), dp_diff_CI=f'[{lo2:+.3f}, {hi2:+.3f}]', p_dp=p2))
ENS = pd.DataFrame(rows); ENS.to_csv(f'{OUT}/background_swap_ensembles.csv', index=False)
print('\nSeed ensembles (seeds 0-2), background swap; Ours minus other, paired bootstrap over lesion groups:'); print(ENS.round(4).to_string(index=False))

# expert structures: seeds 0-2 averaged per image, then the paired Wilcoxon test over lesion groups (validation + test images)
if len(AT_EVAL):
    g_at = meta.group.values[AT_EVAL]; rows = []
    def att_mean(c, k):
        rr = [r for r in PRED if cfg_of(r) == c and (r, 'eval') in ATT]
        return np.nanmean([ATT[(r, 'eval')][k].values for r in rr], 0) if rr else None
    for k in ['excess', 'point_attr', 'hit_attr']:
        a = att_mean('Ours', k)
        for c in ORDER:
            if c == 'Ours' or att_mean(c, k) is None: continue
            b = att_mean(c, k); m = ~np.isnan(a) & ~np.isnan(b)
            ga = pd.Series(a[m]).groupby(g_at[m]).mean(); gb = pd.Series(b[m]).groupby(g_at[m]).mean()
            rows.append(dict(metric=k, versus=c, ours=float(np.nanmean(a[m])), other=float(np.nanmean(b[m])), n_images=int(m.sum()),
                             n_groups=len(ga), p_wilcoxon=float(wilcoxon(ga.values, gb.values).pvalue) if len(ga) > 5 and (ga.values != gb.values).any() else np.nan))
    ATTEST = pd.DataFrame(rows); ATTEST.to_csv(f'{OUT}/expert_structures_tests.csv', index=False)
    print('\nExpert structures (ISIC 2018 Task 2, validation + test images; seeds 0-2 averaged per image; Wilcoxon on lesion groups):')
    print(ATTEST.round(4).to_string(index=False))
    per_attr = {c: {a: float(np.nanmean(np.where(AT['eval'][:, k].reshape(len(AT_EVAL), -1).any(1), att_mean(c, f'hit {a}'), np.nan)))
                    for k, a in enumerate(ATTR)} for c in ORDER if att_mean(c, 'excess') is not None}
    print('\nhot-region share on each structure (images that contain it):'); print(pd.DataFrame(per_attr).round(4).to_string())

G = pd.DataFrame(GEO)
if len(G):
    G['fam'] = G.config.replace(FAM)
    print('\nPrototype source cells on the outer ring of the 7x7 grid / in a corner cell (all runs):')
    print(G.assign(corner=np.isin(G.h, [0, 6]) & np.isin(G.w, [0, 6]), n=1).groupby('fam')[['n', 'ring', 'corner']].sum().to_string())

## 4. Source patches of all prototypes of PPN, the Correia formulation and Ours (for a blinded artifact audit)

In [ ]:
def patch_jpeg(i, h, w, size=192):
    img = np.array(IM[i]).copy(); s = 256 / 224 * 32
    x0, y0 = int(round(w * s)), int(round(h * s)); x1, y1 = int(round((w + 1) * s)), int(round((h + 1) * s))
    cx0, cy0 = max(0, x0 - int(s)), max(0, y0 - int(s)); cx1, cy1 = min(256, x1 + int(s)), min(256, y1 + int(s))
    crop = cv2.resize(img[cy0:cy1, cx0:cx1], (size, size), interpolation=cv2.INTER_CUBIC)
    fx, fy = size / (cx1 - cx0), size / (cy1 - cy0)
    cv2.rectangle(crop, (int((x0 - cx0) * fx), int((y0 - cy0) * fy)), (int((x1 - cx0) * fx) - 1, int((y1 - cy0) * fy) - 1), (0, 255, 255), 2)
    ok, buf = cv2.imencode('.jpg', cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buf.tobytes()).decode()
AUDIT = ['PPN', 'PPN-indep', 'PPN-FA-Correia', 'Correia-indep', 'Ours', 'Ours-indep']
sub = G[G.config.isin(AUDIT)].reset_index(drop=True) if len(G) else pd.DataFrame()
print('patches:', len(sub))
for _, r in sub.iterrows():
    print(f"PATCH\t{r.config}\t{r.run}\t{r.proto}\t{r.les_frac:.3f}\t{r.art_frac:.3f}\t{patch_jpeg(int(r.img), int(r.h), int(r.w))}")
print(f'NB13 finished in {(time.time() - T0) / 60:.1f} min')
print('NB13 FINISHED')